# Magic Hour · Krea 2 Identity Edit

Internal production demo for deterministic head / face swap.

| Input | Role |
| --- | --- |
| **Body** | Scene — pose, clothing, background |
| **Face** | Identity — face / head to transfer |

**Recommended workflow:** edit §1 → **Runtime → Run all**.

**Expected runtime (warm, after models loaded)**

| GPU | Typical wall time |
| --- | --- |
| **A100** (40 GB) | ~1–2 min / image |
| **T4** (16 GB) | ~3–4 min / image |

Cold start adds one-time model load. Weights cache on Drive (~18 GB) so reconnects skip re-download.

Warm re-run after a successful pass: change knobs or re-upload → re-run **§5** (and §6 for preview).

---
## 1 · Settings

Only these knobs are meant to change. Everything else uses production defaults.

To **reproduce** a prior run: open that run’s `run_config.json` and copy the knobs (and optional `PINNED_COMMIT`) below.

In [ ]:
# === User-facing knobs ===
SEED = 46
PROMPT = None                 # None = yaml default prompt
STEPS = 8
CFG = 1.0
OUTPUT_LONG_SIDE = 1024       # final body canvas long side (px)
STITCH = True                 # mask→crop→edit→soft stitch (recommended)
DEBUG = False                 # verbose logs + debug_*.png

# Quality gates (post-run warning thresholds)
IDENTITY_THRESH = 0.35
BODY_PSNR_THRESH = 28.0

# Optional: pin exact repo commit for this session (None = whatever git pull fetched)
PINNED_COMMIT = None          # e.g. "abc1234…"

print("Settings")
print(f"  SEED={SEED}  STEPS={STEPS}  CFG={CFG}  OUTPUT_LONG_SIDE={OUTPUT_LONG_SIDE}")
print(f"  STITCH={STITCH}  DEBUG={DEBUG}")
print(f"  IDENTITY_THRESH={IDENTITY_THRESH}  BODY_PSNR_THRESH={BODY_PSNR_THRESH}")
print(f"  PINNED_COMMIT={PINNED_COMMIT or '(none — use pulled HEAD)'}")
if PROMPT:
    p = str(PROMPT)
    print(f"  PROMPT={p[:80]}…" if len(p) > 80 else f"  PROMPT={p}")
else:
    print("  PROMPT=(yaml default)")

---
## 2 · Setup _(first run / reconnect — collapsed by default)_

Mounts Drive, syncs the repo (optional pin), installs ComfyUI + Krea2 nodes, downloads/verifies weights.

> After the **first** custom-node install on a brand-new runtime:  
> **Runtime → Restart session**, then **Run all** from §1.

In [ ]:
#@title Setup: GPU · Drive · repo · models
from pathlib import Path
import importlib.util
import os
import subprocess

assert Path("/content").exists(), "This notebook is for Google Colab (/content)."

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. Runtime → Change runtime type → GPU (prefer A100), then Run all."
    )
print(f"✓ GPU  {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB)")

from google.colab import drive
print("→ Mounting Google Drive…")
drive.mount("/content/drive")

REPO_URL = "https://github.com/malihashar/headswap_V2.git"
REPO = Path("/content/headswap_V2")
print("→ Syncing repository…")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)

spec = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))

spec_d = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec_d)
spec_d.loader.exec_module(colab_demo)

pin = globals().get("PINNED_COMMIT")
VERSIONS = colab_demo.collect_versions(
    repo=REPO, comfyui=PATHS["comfyui"], pinned_commit=pin
)

print("→ Installing ComfyUI + Krea2 nodes…")
!bash scripts/setup_colab.sh
!bash scripts/setup_krea2_nodes.sh

print("→ Downloading / verifying models…")
!python scripts/download_krea2.py \
  --comfy "$COMFYUI_PATH" \
  --store-dir "$HEADSWAP_MODEL_STORE" \
  --staging-dir "$HEADSWAP_STAGING_DIR" \
  --backend auto \
  --disable-xet

colab_demo.verify_models(PATHS["model_store"], check_sizes=True)
VERSIONS = colab_demo.collect_versions(repo=REPO, comfyui=PATHS["comfyui"])
colab_demo.print_versions(VERSIONS)
colab_demo.ok("Setup ready")
print(
    "\nIf this was the FIRST custom-node install on a fresh runtime: "
    "Runtime → Restart session, then Run all from §1.\n"
)

---
## 3 · Upload body & face

JPG / PNG / WEBP. Faces are validated; if several faces are found, the **largest** is used.

In [ ]:
import importlib.util
from pathlib import Path

from google.colab import files
from IPython.display import display, Markdown

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

custom = REPO / "data" / "custom"
custom.mkdir(parents=True, exist_ok=True)
cache_dir = REPO / ".cache" / "headswap_v2"
cache_dir.mkdir(parents=True, exist_ok=True)

try:
    print("Upload BODY image (scene / clothing / pose)…")
    up_body = files.upload()
    if not up_body:
        raise colab_demo.DemoError("No body image uploaded.")
    body_name = next(iter(up_body))
    body_path = custom / "body.png"
    body_im = colab_demo.save_upload(up_body[body_name], body_path)
    for msg in colab_demo.check_image_geometry(body_im, "Body"):
        colab_demo.warn(msg)
    body_face = colab_demo.require_face(body_im, cache_dir, "body")

    print("Upload FACE image (identity donor)…")
    up_face = files.upload()
    if not up_face:
        raise colab_demo.DemoError("No face image uploaded.")
    face_name = next(iter(up_face))
    face_path = custom / "face.png"
    face_im = colab_demo.save_upload(up_face[face_name], face_path)
    for msg in colab_demo.check_image_geometry(face_im, "Face"):
        colab_demo.warn(msg)
    face_face = colab_demo.require_face(face_im, cache_dir, "face")
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise SystemExit(str(exc))

!python scripts/prepare_eval_set.py --custom data/custom

display(Markdown("### Inputs"))
print(f"Body: {body_face['face_count']} face(s), using largest conf={body_face['confidence']}  size={body_im.size}")
display(body_im.resize((320, 320)))
print(f"Face: {face_face['face_count']} face(s), using largest conf={face_face['confidence']}  size={face_im.size}")
display(face_im.resize((320, 320)))
colab_demo.ok(f"Saved → {body_path} , {face_path}")

---
## 4 · Preflight _(collapsed)_

Confirms GPU, model files + sizes, and detectable faces. Also run automatically inside §5.

In [ ]:
#@title Preflight + environment summary
import importlib.util
from pathlib import Path

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

try:
    if "SEED" not in globals():
        raise colab_demo.DemoError("Run §1 Settings first.")
    VERSIONS = colab_demo.collect_versions(
        repo=REPO, comfyui=PATHS["comfyui"], pinned_commit=globals().get("PINNED_COMMIT")
    )
    from headswap.config import load_config
    _cfg = load_config(REPO / "configs" / "krea2_identity_edit.yaml")
    _prompt = PROMPT if PROMPT else _cfg.get("prompt")
    PARAMS = {
        "seed": SEED,
        "steps": STEPS,
        "cfg": CFG,
        "output_long_side": OUTPUT_LONG_SIDE,
        "stitch": STITCH,
        "debug": DEBUG,
        "identity_thresh": IDENTITY_THRESH,
        "body_psnr_thresh": BODY_PSNR_THRESH,
        "prompt": _prompt,
        "repo_commit": (VERSIONS.get("git") or {}).get("commit"),
    }
    PREFLIGHT = colab_demo.preflight(
        body_path=REPO / "data" / "custom" / "body.png",
        face_path=REPO / "data" / "custom" / "face.png",
        cache_dir=REPO / ".cache" / "headswap_v2",
        model_store=PATHS["model_store"],
    )
    colab_demo.environment_summary(paths=PATHS, versions=VERSIONS, params=PARAMS)
except colab_demo.DemoError as exc:
    colab_demo.fail(str(exc))
    raise SystemExit(str(exc))

---
## 5 · Run inference

Models load once and stay warm. Each run writes a timestamped package under `/content/headswap_outputs/run_YYYYMMDD_HHMMSS/`.

In [ ]:
import importlib.util
import json
import time
import traceback
from pathlib import Path

from PIL import Image

REPO = Path("/content/headswap_V2")
spec_env = importlib.util.spec_from_file_location("colab_env", REPO / "scripts" / "colab_env.py")
colab_env = importlib.util.module_from_spec(spec_env)
spec_env.loader.exec_module(colab_env)
PATHS = colab_env.apply_env(colab_env.default_paths(use_drive=True))

spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

from headswap.config import load_config
from headswap.pipelines import create_pipeline
from headswap.pipelines.krea2 import get_shared_krea2_runtime

CONFIG = REPO / "configs" / "krea2_identity_edit.yaml"
BODY_PATH = REPO / "data" / "custom" / "body.png"
FACE_PATH = REPO / "data" / "custom" / "face.png"
CACHE = REPO / ".cache" / "headswap_v2"

RUN_OK = False
RUN_ERROR = None
RESULT_PATH = None
STABLE_PATH = None
RUN_DIR = None
META = {}
QUALITY = {}

try:
    if "SEED" not in globals():
        raise colab_demo.DemoError("Run §1 Settings first.")

    VERSIONS = colab_demo.collect_versions(
        repo=REPO, comfyui=PATHS["comfyui"], pinned_commit=globals().get("PINNED_COMMIT")
    )
    PREFLIGHT = colab_demo.preflight(
        body_path=BODY_PATH,
        face_path=FACE_PATH,
        cache_dir=CACHE,
        model_store=PATHS["model_store"],
    )

    base_cfg = load_config(CONFIG)
    cfg = colab_demo.apply_user_knobs(
        base_cfg,
        seed=SEED,
        prompt=PROMPT,
        steps=STEPS,
        cfg_scale=CFG,
        output_long_side=OUTPUT_LONG_SIDE,
        stitch=STITCH,
        debug=DEBUG,
    )
    effective_prompt = cfg.get("prompt")
    PARAMS = {
        "seed": SEED,
        "steps": int(cfg["steps"]),
        "cfg": float(cfg["cfg"]),
        "output_long_side": OUTPUT_LONG_SIDE,
        "stitch": STITCH,
        "debug": DEBUG,
        "identity_thresh": IDENTITY_THRESH,
        "body_psnr_thresh": BODY_PSNR_THRESH,
        "prompt": effective_prompt,
        "repo_commit": (VERSIONS.get("git") or {}).get("commit"),
    }
    colab_demo.print_run_parameters(PARAMS)

    if "_KREA2_PIPE" not in globals() or _KREA2_PIPE is None:
        colab_demo.progress("Loading models (cold start)…")
        rt = get_shared_krea2_runtime(init_custom_nodes=True)
        _KREA2_PIPE = create_pipeline(cfg, runtime=rt)
    else:
        colab_demo.progress("Reusing warm model cache…")
        _KREA2_PIPE.cfg.update(cfg)

    body = Image.open(BODY_PATH).convert("RGB")
    face = Image.open(FACE_PATH).convert("RGB")

    pair_scratch = REPO / "results" / "krea2_identity_edit" / "images" / "custom_001"
    pair_scratch.mkdir(parents=True, exist_ok=True)

    colab_demo.progress("Running head swap…")
    t0 = time.perf_counter()
    if DEBUG:
        result = _KREA2_PIPE.run(body, face, out_dir=pair_scratch)
    else:
        with colab_demo.quiet_logs(debug=False):
            result = _KREA2_PIPE.run(body, face, out_dir=pair_scratch)
    wall = colab_demo.elapsed(t0)
    META = dict(result.meta or {})
    timing = META.get("timing_s") or {}

    colab_demo.progress("Scoring quality…")
    QUALITY = colab_demo.score_result(
        body=body,
        face=face,
        result=result.image,
        latency_s=float(result.latency_s),
        cache_dir=CACHE,
        identity_thresh=IDENTITY_THRESH,
        body_psnr_thresh=BODY_PSNR_THRESH,
        stitch=STITCH,
    )

    RUN_DIR = colab_demo.make_run_dir(PATHS["outputs"])
    run_config = {
        "knobs": PARAMS,
        "pipeline_cfg": {
            "config": str(CONFIG),
            "seed": META.get("seed"),
            "steps": META.get("steps"),
            "cfg": META.get("cfg"),
            "mask_crop_stitch": META.get("mask_crop_stitch"),
            "prompt": META.get("prompt"),
            "scene_size": META.get("scene_size"),
            "body_size": META.get("body_size"),
        },
        "versions": VERSIONS,
        "models": PREFLIGHT.get("models"),
        "faces": {"body": PREFLIGHT.get("body"), "face": PREFLIGHT.get("face")},
        "reproduce": {
            "instructions": (
                "Set §1 knobs from knobs, optionally PINNED_COMMIT=versions.git.commit, "
                "re-upload the same body/face, Run all. Same seed + same inputs ⇒ same sampling path."
            ),
            "knobs_from_config": colab_demo.knobs_from_run_config(
                {"knobs": PARAMS, "versions": VERSIONS}
            ),
        },
    }
    metrics_doc = {
        "pair_id": "custom_001",
        "success": bool(QUALITY.get("success")),
        "latency_s": float(result.latency_s),
        "wall_s": wall,
        "quality": QUALITY,
        "meta": META,
    }
    paths_written = colab_demo.save_output_package(
        RUN_DIR,
        result_image=result.image,
        run_config=run_config,
        metrics=metrics_doc,
        timing=timing,
        debug_paths=dict(result.debug_paths or {}),
        save_debug=bool(DEBUG),
    )
    RESULT_PATH = Path(paths_written["result"])
    STABLE_PATH = Path(paths_written["stable"])

    # Mirror into results/ for harness familiarity
    harness = REPO / "results" / "krea2_identity_edit"
    harness.mkdir(parents=True, exist_ok=True)
    (harness / "metrics.json").write_text(json.dumps(metrics_doc, indent=2, default=str))

    colab_demo.print_run_summary(
        success=True,
        total_s=wall,
        sampling_s=timing.get("diffusion_sampling"),
        steps=META.get("steps"),
        seed=META.get("seed"),
        gpu=(PREFLIGHT.get("gpu") or {}).get("name"),
        resolution=result.image.size,
        output_path=RUN_DIR,
        cache_hit=META.get("model_cache_hit"),
        quality=QUALITY,
    )
    print(f"  package           {RUN_DIR}")
    RUN_OK = True

except colab_demo.DemoError as exc:
    RUN_ERROR = str(exc)
    colab_demo.print_run_summary(
        success=False, total_s=0.0, sampling_s=None, steps=None,
        seed=globals().get("SEED"), gpu=None, resolution=None,
        output_path=None, error=RUN_ERROR,
    )
except Exception as exc:
    RUN_ERROR = str(exc)
    colab_demo.fail(f"Unexpected failure: {exc}")
    if globals().get("DEBUG"):
        traceback.print_exc()
    else:
        print("Set DEBUG=True in §1 and re-run for a full traceback.")
    colab_demo.print_run_summary(
        success=False, total_s=0.0, sampling_s=None, steps=None,
        seed=globals().get("SEED"), gpu=None, resolution=None,
        output_path=None, error=RUN_ERROR,
    )

---
## 6 · Results

Preview + download. Debug intermediates only when `DEBUG=True`.

In [ ]:
import importlib.util
from pathlib import Path

from google.colab import files
from IPython.display import display, Markdown
from PIL import Image

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

if not globals().get("RUN_OK"):
    colab_demo.fail(globals().get("RUN_ERROR") or "No successful run yet — fix §5 and re-run.")
else:
    body = Image.open(REPO / "data" / "custom" / "body.png").convert("RGB")
    face = Image.open(REPO / "data" / "custom" / "face.png").convert("RGB")
    result = Image.open(RESULT_PATH).convert("RGB")

    display(Markdown("### Body · Face · Result"))
    display(colab_demo.show_side_by_side(body, face, result))

    if QUALITY:
        display(Markdown("### Quality"))
        colab_demo.print_quality_report(QUALITY)

    if globals().get("DEBUG") and RUN_DIR is not None:
        display(Markdown("### Debug intermediates"))
        dbg = Path(RUN_DIR) / "debug"
        shown = 0
        if dbg.is_dir():
            for p in sorted(dbg.glob("*.png")):
                print(p.name)
                display(Image.open(p).convert("RGB").resize((256, 256)))
                shown += 1
        if not shown:
            print("(no debug assets in this package)")

    download_path = Path(STABLE_PATH) if STABLE_PATH else Path(RESULT_PATH)
    display(Markdown("### Download"))
    print(f"Package: {RUN_DIR}")
    print(f"Image:   {download_path}")
    try:
        files.download(str(download_path))
    except Exception as exc:
        print(f"Browser download helper failed ({exc}). Open the path above in the file browser.")

---
## 7 · Run summary

Stakeholder card — the numbers to screenshot or paste without scrolling logs.


In [ ]:
import importlib.util
import json
from pathlib import Path

REPO = Path("/content/headswap_V2")
spec = importlib.util.spec_from_file_location("colab_demo", REPO / "scripts" / "colab_demo.py")
colab_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(colab_demo)

if not globals().get("RUN_OK"):
    colab_demo.fail(globals().get("RUN_ERROR") or "No successful run yet — fix §5 and re-run.")
else:
    timing = (META or {}).get("timing_s") or {}
    quality = QUALITY or {}
    versions = globals().get("VERSIONS") or {}
    commit = (versions.get("git") or {}).get("commit_short")
    gpu = None
    if globals().get("PREFLIGHT"):
        gpu = (PREFLIGHT.get("gpu") or {}).get("name")
    gpu = gpu or (versions.get("torch") or {}).get("gpu")

    total_s = None
    if RUN_DIR is not None:
        metrics_path = Path(RUN_DIR) / "metrics.json"
        if metrics_path.is_file():
            total_s = json.loads(metrics_path.read_text()).get("wall_s")
    if total_s is None:
        total_s = (META or {}).get("latency_s")

    colab_demo.print_eval_card(
        gpu=gpu,
        commit=commit,
        identity=quality.get("identity_cosine"),
        body_psnr=quality.get("body_preserve_psnr"),
        sampling_s=timing.get("diffusion_sampling"),
        total_s=float(total_s) if total_s is not None else None,
        output_path=STABLE_PATH or RESULT_PATH,
        success=quality.get("success"),
    )


---
## Notes

| Topic | Behavior |
| --- | --- |
| Run all | Preferred first-time path |
| Reproduce | Copy knobs (+ optional `PINNED_COMMIT`) from `run_*/run_config.json` |
| Output package | `result.png`, `run_config.json`, `metrics.json`, `timing.json`, optional `debug/` |
| Stable shortcut | `/content/headswap_outputs/HEADSWAP_RESULT.png` |
| Multi-face | Largest face is selected; warning printed |
| Run summary | §7 stakeholder card (identity / PSNR / timing / output) |
| Docs | `COLAB.md` |

*Magic Hour · headswap_V2 internal demo*
